# HeartSense V3 — Feature Sufficiency, Robustness & Calibration

This experiment asks a stronger research question than another stacking iteration: **are the original six HeartSense predictors sufficient, or do the additional five variables provide a reproducible and more robust gain?**

It compares Core-6, Core-8, Core-9 and Extended-11 under identical outer cross-validation folds, and adds robustness, calibration, decision-curve and paired statistical analyses.

Run all cells. Results are written to `HeartSense_V3_Results/` and zipped as `HeartSense_V3_Results_Bundle.zip`.


In [ ]:

# ======================================================================
# HeartSense V3 — Feature Sufficiency, Robustness & Calibration Benchmark
# ======================================================================
#
# RESEARCH QUESTION
# -----------------
# Does the original HeartSense Core-6 feature set contain enough predictive
# information, or do the additional five clinical variables in the same
# 918-row dataset produce a reproducible and robust gain?
#
# FEATURE TIERS
# -------------
# Core-6:
#   Age, Sex, ChestPainType, RestingBP, MaxHR, ExerciseAngina
#
# Core-8:
#   Core-6 + Cholesterol + FastingBS
#
# Core-9:
#   Core-8 + RestingECG
#
# Extended-11:
#   Core-9 + Oldpeak + ST_Slope
#
# VALIDATION
# ----------
# - 5-fold stratified OUTER CV
# - identical outer folds for every feature tier and model
# - every record is predicted only by a model that did not train on it
# - bootstrap 95% confidence intervals
# - paired bootstrap comparisons between Core-6 and Extended-11
#
# ROBUSTNESS
# ----------
# Models trained on clean outer-training data are also tested on:
# - 10% feature missingness
# - 20% feature missingness
# - mild numeric measurement noise
# Missingness/noise are injected into OUTER TEST data only.
#
# NOTE
# ----
# This is internal validation on the public 918-row dataset.
# It does not claim external clinical validation or ECG/IoT fusion.

import os, sys, json, random, shutil, warnings, subprocess, importlib
from pathlib import Path

warnings.filterwarnings("ignore")

REQ = {
    "numpy":"numpy",
    "pandas":"pandas",
    "sklearn":"scikit-learn",
    "scipy":"scipy",
    "matplotlib":"matplotlib",
    "catboost":"catboost",
    "lightgbm":"lightgbm",
    "xgboost":"xgboost",
    "statsmodels":"statsmodels",
}
missing=[]
for m,p in REQ.items():
    try:
        importlib.import_module(m)
    except Exception:
        missing.append(p)
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable,"-m","pip","install","-q"]+missing)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve

# -----------------------
# 1) Configuration
# -----------------------
SEED=42
OUTER_FOLDS=5
BOOTSTRAP_REPS=2000
ROBUST_REPEATS=5

np.random.seed(SEED)
random.seed(SEED)

TARGET="HeartDisease"

FEATURE_TIERS = {
    "Core_6": [
        "Age","Sex","ChestPainType","RestingBP","MaxHR","ExerciseAngina"
    ],
    "Core_8": [
        "Age","Sex","ChestPainType","RestingBP","MaxHR","ExerciseAngina",
        "Cholesterol","FastingBS"
    ],
    "Core_9": [
        "Age","Sex","ChestPainType","RestingBP","MaxHR","ExerciseAngina",
        "Cholesterol","FastingBS","RestingECG"
    ],
    "Extended_11": [
        "Age","Sex","ChestPainType","RestingBP","MaxHR","ExerciseAngina",
        "Cholesterol","FastingBS","RestingECG","Oldpeak","ST_Slope"
    ]
}

NUMERIC_ALL = ["Age","RestingBP","MaxHR","Cholesterol","Oldpeak"]
CATEGORICAL_ALL = [
    "Sex","ChestPainType","ExerciseAngina","FastingBS","RestingECG","ST_Slope"
]

WORKDIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ROOT = WORKDIR/"HeartSense_V3_Results"
if ROOT.exists():
    shutil.rmtree(ROOT)

for d in [
    "01_Audit",
    "02_OuterOOF_Predictions",
    "03_Main_Metrics",
    "04_FeatureTier_Comparison",
    "05_Robustness",
    "06_Calibration_DCA",
    "07_Statistics",
    "08_Figures",
    "09_Reproducibility",
]:
    (ROOT/d).mkdir(parents=True,exist_ok=True)

# -----------------------
# 2) Dataset discovery
# -----------------------
def locate_heart_csv():
    roots=[Path.cwd()]
    if Path("/kaggle/input").exists():
        roots.insert(0,Path("/kaggle/input"))

    expected=set(FEATURE_TIERS["Extended_11"]+[TARGET])

    for root in roots:
        for p in root.rglob("heart.csv"):
            try:
                sample=pd.read_csv(p,nrows=5)
                if expected.issubset(sample.columns):
                    full=pd.read_csv(p)
                    if len(full)==918:
                        return p
            except Exception:
                pass

    # KaggleHub fallback
    try:
        import kagglehub
    except Exception:
        try:
            subprocess.check_call([sys.executable,"-m","pip","install","-q","kagglehub"])
            import kagglehub
        except Exception:
            kagglehub=None

    if kagglehub is not None:
        folder=Path(kagglehub.dataset_download("fedesoriano/heart-failure-prediction"))
        for p in folder.rglob("heart.csv"):
            sample=pd.read_csv(p,nrows=5)
            if expected.issubset(sample.columns):
                return p

    raise FileNotFoundError(
        "heart.csv not found. Attach the Kaggle Heart Failure Prediction Dataset (fedesoriano)."
    )

DATA_PATH=locate_heart_csv()
raw=pd.read_csv(DATA_PATH)
df=raw.drop_duplicates().reset_index(drop=True)

if TARGET not in df.columns:
    raise ValueError("Target column HeartDisease not found.")

y=df[TARGET].astype(int).copy()

audit_rows=[
    ["dataset_path",str(DATA_PATH)],
    ["rows_loaded",len(raw)],
    ["rows_after_exact_full_duplicate_removal",len(df)],
    ["full_row_duplicates_removed",len(raw)-len(df)],
    ["target_positive",int(y.sum())],
    ["target_negative",int((1-y).sum())],
]
for tier,cols in FEATURE_TIERS.items():
    audit_rows.append([f"{tier}_missing_cells",int(df[cols].isna().sum().sum())])

pd.DataFrame(audit_rows,columns=["Item","Value"]).to_csv(
    ROOT/"01_Audit"/"data_audit.csv",index=False
)

# -----------------------
# 3) Helpers
# -----------------------
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore",sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore",sparse=False)

def build_preprocessor(features):
    num=[c for c in features if c in NUMERIC_ALL]
    cat=[c for c in features if c in CATEGORICAL_ALL]

    num_pipe=Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())
    ])
    cat_pipe=Pipeline([
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("onehot",make_ohe())
    ])

    return ColumnTransformer([
        ("num",num_pipe,num),
        ("cat",cat_pipe,cat)
    ])

def make_models(features):
    prep=build_preprocessor(features)
    return {
        "Logistic_Regression":Pipeline([
            ("prep",clone(prep)),
            ("model",LogisticRegression(max_iter=2500,random_state=SEED))
        ]),
        "Gradient_Boosting":Pipeline([
            ("prep",clone(prep)),
            ("model",GradientBoostingClassifier(
                n_estimators=250,learning_rate=0.03,max_depth=2,random_state=SEED
            ))
        ]),
        "Random_Forest":Pipeline([
            ("prep",clone(prep)),
            ("model",RandomForestClassifier(
                n_estimators=600,min_samples_leaf=2,max_features="sqrt",
                random_state=SEED,n_jobs=-1
            ))
        ]),
        "XGBoost":Pipeline([
            ("prep",clone(prep)),
            ("model",XGBClassifier(
                n_estimators=450,max_depth=3,learning_rate=0.03,
                subsample=0.9,colsample_bytree=0.9,
                reg_lambda=1.0,eval_metric="logloss",
                random_state=SEED,n_jobs=-1
            ))
        ]),
        "LightGBM":Pipeline([
            ("prep",clone(prep)),
            ("model",LGBMClassifier(
                n_estimators=500,learning_rate=0.025,num_leaves=15,
                min_child_samples=20,reg_lambda=1.0,
                random_state=SEED,n_jobs=-1,verbosity=-1
            ))
        ]),
    }

CATBOOST_PARAMS=dict(
    iterations=700,
    depth=6,
    learning_rate=0.035,
    l2_leaf_reg=1.0,
    random_strength=0.05,
    bagging_temperature=1.0,
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=0,
    random_seed=SEED,
    allow_writing_files=False
)

def cb_prep(train_df, apply_df, features):
    num=[c for c in features if c in NUMERIC_ALL]
    cat=[c for c in features if c in CATEGORICAL_ALL]
    tr=train_df.copy()
    ap=apply_df.copy()

    for c in num:
        med=pd.to_numeric(tr[c],errors="coerce").median()
        tr[c]=pd.to_numeric(tr[c],errors="coerce").fillna(med)
        ap[c]=pd.to_numeric(ap[c],errors="coerce").fillna(med)

    for c in cat:
        mode=tr[c].astype("string").dropna().mode()
        fill=str(mode.iloc[0]) if len(mode) else "Missing"
        tr[c]=tr[c].astype("string").fillna(fill).astype(str)
        ap[c]=ap[c].astype("string").fillna(fill).astype(str)

    return tr,ap,cat

def metric_dict(ytrue,prob,threshold=0.5):
    prob=np.asarray(prob,dtype=float)
    pred=(prob>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(ytrue,pred,labels=[0,1]).ravel()
    return {
        "Accuracy":accuracy_score(ytrue,pred),
        "Precision_PPV":precision_score(ytrue,pred,zero_division=0),
        "Sensitivity_Recall":recall_score(ytrue,pred,zero_division=0),
        "Specificity":tn/(tn+fp) if tn+fp else np.nan,
        "NPV":tn/(tn+fn) if tn+fn else np.nan,
        "F1":f1_score(ytrue,pred,zero_division=0),
        "Balanced_Accuracy":balanced_accuracy_score(ytrue,pred),
        "MCC":matthews_corrcoef(ytrue,pred),
        "ROC_AUC":roc_auc_score(ytrue,prob),
        "PR_AUC":average_precision_score(ytrue,prob),
        "Brier":brier_score_loss(ytrue,prob)
    }

def inject_missingness(frame, fraction, rng):
    out=frame.copy()
    mask=rng.random(out.shape)<fraction
    out=out.mask(mask)
    return out

def inject_numeric_noise(frame, train_frame, features, rng, scale_fraction=0.05):
    out=frame.copy()
    for c in [x for x in features if x in NUMERIC_ALL]:
        sd=pd.to_numeric(train_frame[c],errors="coerce").std()
        if np.isfinite(sd) and sd>0:
            noise=rng.normal(0,scale_fraction*sd,size=len(out))
            out[c]=pd.to_numeric(out[c],errors="coerce")+noise
    return out

def bootstrap_ci(ytrue,prob,reps=BOOTSTRAP_REPS,seed=SEED):
    ytrue=np.asarray(ytrue)
    prob=np.asarray(prob)
    rng=np.random.default_rng(seed)
    keys=list(metric_dict(ytrue,prob).keys())
    store={k:[] for k in keys}
    for _ in range(reps):
        idx=rng.integers(0,len(ytrue),len(ytrue))
        if len(np.unique(ytrue[idx]))<2:
            continue
        m=metric_dict(ytrue[idx],prob[idx])
        for k,v in m.items():
            if np.isfinite(v):
                store[k].append(v)
    point=metric_dict(ytrue,prob)
    rows=[]
    for k in keys:
        vals=np.asarray(store[k])
        rows.append({
            "Metric":k,
            "Estimate":point[k],
            "CI95_Low":np.percentile(vals,2.5),
            "CI95_High":np.percentile(vals,97.5)
        })
    return pd.DataFrame(rows)

# -----------------------
# 4) Outer OOF evaluation
# -----------------------
outer=StratifiedKFold(n_splits=OUTER_FOLDS,shuffle=True,random_state=SEED)

MODEL_NAMES=[
    "Logistic_Regression","Gradient_Boosting","Random_Forest",
    "XGBoost","LightGBM","CatBoost"
]

# clean OOF probabilities
oof={
    tier:{m:np.full(len(df),np.nan) for m in MODEL_NAMES}
    for tier in FEATURE_TIERS
}

# robustness: list of fold/repeat metrics rather than pooled predictions
robust_rows=[]
outer_metric_rows=[]

for fold,(tr_idx,te_idx) in enumerate(outer.split(df,y),start=1):
    print(f"Outer fold {fold}/{OUTER_FOLDS}")
    ytr=y.iloc[tr_idx]
    yte=y.iloc[te_idx]

    for tier,features in FEATURE_TIERS.items():
        Xall=df[features].copy()
        Xtr=Xall.iloc[tr_idx].copy()
        Xte=Xall.iloc[te_idx].copy()

        models=make_models(features)

        # -------- sklearn-family models --------
        for name,model in models.items():
            model.fit(Xtr,ytr)
            p=model.predict_proba(Xte)[:,1]
            oof[tier][name][te_idx]=p

            row={"OuterFold":fold,"FeatureTier":tier,"Model":name,"Condition":"Clean"}
            row.update(metric_dict(yte,p))
            outer_metric_rows.append(row)

            # Robustness
            for rep in range(ROBUST_REPEATS):
                rng=np.random.default_rng(SEED+fold*1000+rep*100+len(features))

                for frac,label in [(0.10,"Missing_10pct"),(0.20,"Missing_20pct")]:
                    Xpert=inject_missingness(Xte,frac,rng)
                    pp=model.predict_proba(Xpert)[:,1]
                    rr={"OuterFold":fold,"Repeat":rep,"FeatureTier":tier,"Model":name,"Condition":label}
                    rr.update(metric_dict(yte,pp))
                    robust_rows.append(rr)

                Xnoise=inject_numeric_noise(Xte,Xtr,features,rng,scale_fraction=0.05)
                pp=model.predict_proba(Xnoise)[:,1]
                rr={"OuterFold":fold,"Repeat":rep,"FeatureTier":tier,"Model":name,"Condition":"NumericNoise_5pctSD"}
                rr.update(metric_dict(yte,pp))
                robust_rows.append(rr)

        # -------- CatBoost --------
        Xtr_cb,Xte_cb,catcols=cb_prep(Xtr,Xte,features)
        cb=CatBoostClassifier(**CATBOOST_PARAMS)
        cb.fit(Xtr_cb,ytr,cat_features=catcols)
        p=cb.predict_proba(Xte_cb)[:,1]
        oof[tier]["CatBoost"][te_idx]=p

        row={"OuterFold":fold,"FeatureTier":tier,"Model":"CatBoost","Condition":"Clean"}
        row.update(metric_dict(yte,p))
        outer_metric_rows.append(row)

        # Robustness for CatBoost: perturb raw test then apply train-derived prep
        for rep in range(ROBUST_REPEATS):
            rng=np.random.default_rng(SEED+fold*1000+rep*100+len(features)+777)

            for frac,label in [(0.10,"Missing_10pct"),(0.20,"Missing_20pct")]:
                Xpert=inject_missingness(Xte,frac,rng)
                _,Xpert_cb,_=cb_prep(Xtr,Xpert,features)
                pp=cb.predict_proba(Xpert_cb)[:,1]
                rr={"OuterFold":fold,"Repeat":rep,"FeatureTier":tier,"Model":"CatBoost","Condition":label}
                rr.update(metric_dict(yte,pp))
                robust_rows.append(rr)

            Xnoise=inject_numeric_noise(Xte,Xtr,features,rng,scale_fraction=0.05)
            _,Xnoise_cb,_=cb_prep(Xtr,Xnoise,features)
            pp=cb.predict_proba(Xnoise_cb)[:,1]
            rr={"OuterFold":fold,"Repeat":rep,"FeatureTier":tier,"Model":"CatBoost","Condition":"NumericNoise_5pctSD"}
            rr.update(metric_dict(yte,pp))
            robust_rows.append(rr)

outer_metrics=pd.DataFrame(outer_metric_rows)
outer_metrics.to_csv(
    ROOT/"03_Main_Metrics"/"outer_fold_clean_metrics.csv",index=False
)

robust_df=pd.DataFrame(robust_rows)
robust_df.to_csv(
    ROOT/"05_Robustness"/"robustness_all_runs.csv",index=False
)

# -----------------------
# 5) Pooled clean OOF metrics + CIs
# -----------------------
pooled_rows=[]
ci_rows=[]
pred_table=pd.DataFrame({"Actual":y.values})

for tier in FEATURE_TIERS:
    for model in MODEL_NAMES:
        prob=oof[tier][model]
        pred_table[f"{tier}__{model}__Probability"]=prob

        m=metric_dict(y,prob)
        pooled_rows.append({"FeatureTier":tier,"Model":model,**m})

        ci=bootstrap_ci(y.values,prob)
        ci.insert(0,"Model",model)
        ci.insert(0,"FeatureTier",tier)
        ci_rows.append(ci)

pooled=pd.DataFrame(pooled_rows)
pooled.to_csv(ROOT/"03_Main_Metrics"/"pooled_clean_OOF_metrics.csv",index=False)
pd.concat(ci_rows,ignore_index=True).to_csv(
    ROOT/"03_Main_Metrics"/"pooled_clean_OOF_metrics_95CI.csv",index=False
)
pred_table.to_csv(
    ROOT/"02_OuterOOF_Predictions"/"all_clean_OOF_predictions.csv",index=False
)

# -----------------------
# 6) Feature-tier comparison
# -----------------------
tier_summary=(
    pooled.groupby("FeatureTier")[["Accuracy","Balanced_Accuracy","ROC_AUC","PR_AUC","Brier"]]
    .mean()
    .reset_index()
)
tier_summary.to_csv(
    ROOT/"04_FeatureTier_Comparison"/"mean_performance_by_feature_tier.csv",index=False
)

# Per-model Core6 vs Extended11 differences with paired bootstrap
paired_rows=[]
rng=np.random.default_rng(SEED+999)

for model in MODEL_NAMES:
    p6=oof["Core_6"][model]
    p11=oof["Extended_11"][model]
    yt=y.values

    point6=metric_dict(yt,p6)
    point11=metric_dict(yt,p11)

    auc_diffs=[]
    acc_diffs=[]
    bal_diffs=[]
    brier_diffs=[]

    for _ in range(3000):
        idx=rng.integers(0,len(yt),len(yt))
        if len(np.unique(yt[idx]))<2:
            continue

        m6=metric_dict(yt[idx],p6[idx])
        m11=metric_dict(yt[idx],p11[idx])

        auc_diffs.append(m11["ROC_AUC"]-m6["ROC_AUC"])
        acc_diffs.append(m11["Accuracy"]-m6["Accuracy"])
        bal_diffs.append(m11["Balanced_Accuracy"]-m6["Balanced_Accuracy"])
        # Lower Brier is better, so positive means E11 improved
        brier_diffs.append(m6["Brier"]-m11["Brier"])

    paired_rows.append({
        "Model":model,
        "Core6_Accuracy":point6["Accuracy"],
        "Extended11_Accuracy":point11["Accuracy"],
        "Accuracy_Difference_E11_minus_C6":point11["Accuracy"]-point6["Accuracy"],
        "Accuracy_Diff_CI95_Low":np.percentile(acc_diffs,2.5),
        "Accuracy_Diff_CI95_High":np.percentile(acc_diffs,97.5),

        "Core6_BalancedAccuracy":point6["Balanced_Accuracy"],
        "Extended11_BalancedAccuracy":point11["Balanced_Accuracy"],
        "BalancedAccuracy_Difference_E11_minus_C6":
            point11["Balanced_Accuracy"]-point6["Balanced_Accuracy"],
        "BalancedAccuracy_Diff_CI95_Low":np.percentile(bal_diffs,2.5),
        "BalancedAccuracy_Diff_CI95_High":np.percentile(bal_diffs,97.5),

        "Core6_ROC_AUC":point6["ROC_AUC"],
        "Extended11_ROC_AUC":point11["ROC_AUC"],
        "AUC_Difference_E11_minus_C6":point11["ROC_AUC"]-point6["ROC_AUC"],
        "AUC_Diff_CI95_Low":np.percentile(auc_diffs,2.5),
        "AUC_Diff_CI95_High":np.percentile(auc_diffs,97.5),

        "Core6_Brier":point6["Brier"],
        "Extended11_Brier":point11["Brier"],
        "Brier_Improvement_C6_minus_E11":point6["Brier"]-point11["Brier"],
        "Brier_Improvement_CI95_Low":np.percentile(brier_diffs,2.5),
        "Brier_Improvement_CI95_High":np.percentile(brier_diffs,97.5),
    })

paired=pd.DataFrame(paired_rows)
paired.to_csv(
    ROOT/"07_Statistics"/"Core6_vs_Extended11_paired_bootstrap.csv",index=False
)

# -----------------------
# 7) Robustness summaries
# -----------------------
clean_reference=pooled[
    ["FeatureTier","Model","Accuracy","Balanced_Accuracy","ROC_AUC","Brier"]
].rename(columns={
    "Accuracy":"Clean_Accuracy",
    "Balanced_Accuracy":"Clean_Balanced_Accuracy",
    "ROC_AUC":"Clean_ROC_AUC",
    "Brier":"Clean_Brier"
})

robust_summary=(
    robust_df.groupby(["FeatureTier","Model","Condition"])[
        ["Accuracy","Balanced_Accuracy","ROC_AUC","Brier"]
    ]
    .mean()
    .reset_index()
    .merge(clean_reference,on=["FeatureTier","Model"],how="left")
)

robust_summary["Accuracy_Drop"]=robust_summary["Clean_Accuracy"]-robust_summary["Accuracy"]
robust_summary["BalancedAccuracy_Drop"]=(
    robust_summary["Clean_Balanced_Accuracy"]-robust_summary["Balanced_Accuracy"]
)
robust_summary["AUC_Drop"]=robust_summary["Clean_ROC_AUC"]-robust_summary["ROC_AUC"]
robust_summary["Brier_Worsening"]=robust_summary["Brier"]-robust_summary["Clean_Brier"]

robust_summary.to_csv(
    ROOT/"05_Robustness"/"robustness_summary.csv",index=False
)

# -----------------------
# 8) Identify best Extended-11 model and produce calibration/DCA
# -----------------------
extended=pooled[pooled["FeatureTier"]=="Extended_11"].copy()
best_extended=extended.sort_values(
    ["ROC_AUC","Balanced_Accuracy"],ascending=False
).iloc[0]
BEST_MODEL=str(best_extended["Model"])
best_prob=oof["Extended_11"][BEST_MODEL]

# Calibration curve
frac,meanp=calibration_curve(y,best_prob,n_bins=10,strategy="quantile")
cal_df=pd.DataFrame({
    "MeanPredictedProbability":meanp,
    "ObservedFraction":frac
})
cal_df.to_csv(
    ROOT/"06_Calibration_DCA"/f"{BEST_MODEL}_Extended11_calibration_points.csv",
    index=False
)

pclip=np.clip(best_prob,1e-6,1-1e-6)
logit=np.log(pclip/(1-pclip))
try:
    fit=sm.Logit(y.values,sm.add_constant(logit)).fit(disp=0)
    cal_intercept=float(fit.params[0])
    cal_slope=float(fit.params[1])
except Exception:
    cal_intercept=np.nan
    cal_slope=np.nan

pd.DataFrame([{
    "FeatureTier":"Extended_11",
    "Model":BEST_MODEL,
    "Brier":brier_score_loss(y,best_prob),
    "Calibration_Intercept":cal_intercept,
    "Calibration_Slope":cal_slope
}]).to_csv(
    ROOT/"06_Calibration_DCA"/"best_extended_calibration_metrics.csv",
    index=False
)

fig,ax=plt.subplots(figsize=(6,5))
ax.plot(meanp,frac,marker="o",label=f"Extended-11 {BEST_MODEL}")
ax.plot([0,1],[0,1],linestyle="--",label="Ideal")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed event fraction")
ax.set_title("Calibration — Best Extended-11 Model")
ax.legend()
fig.tight_layout()
fig.savefig(
    ROOT/"08_Figures"/"best_extended_calibration.png",
    dpi=300,bbox_inches="tight"
)
plt.close(fig)

# Decision curve
def decision_curve(ytrue,prob,thresholds):
    ytrue=np.asarray(ytrue)
    prob=np.asarray(prob)
    n=len(ytrue)
    prev=ytrue.mean()
    rows=[]
    for t in thresholds:
        pred=prob>=t
        tp=np.sum((pred==1)&(ytrue==1))
        fp=np.sum((pred==1)&(ytrue==0))
        nb_model=tp/n - fp/n*(t/(1-t))
        nb_all=prev - (1-prev)*(t/(1-t))
        rows.append([t,nb_model,nb_all,0.0])
    return pd.DataFrame(
        rows,
        columns=["Threshold","Model_NetBenefit","TreatAll_NetBenefit","TreatNone_NetBenefit"]
    )

dca=decision_curve(y.values,best_prob,np.linspace(0.05,0.80,31))
dca.to_csv(
    ROOT/"06_Calibration_DCA"/"best_extended_decision_curve.csv",index=False
)

fig,ax=plt.subplots(figsize=(7,5))
ax.plot(dca["Threshold"],dca["Model_NetBenefit"],label=f"Extended-11 {BEST_MODEL}")
ax.plot(dca["Threshold"],dca["TreatAll_NetBenefit"],label="Treat all")
ax.plot(dca["Threshold"],dca["TreatNone_NetBenefit"],label="Treat none")
ax.set_xlabel("Threshold")
ax.set_ylabel("Net benefit")
ax.set_title("Decision Curve Analysis")
ax.legend()
fig.tight_layout()
fig.savefig(
    ROOT/"08_Figures"/"best_extended_decision_curve.png",
    dpi=300,bbox_inches="tight"
)
plt.close(fig)

# -----------------------
# 9) Figures: Core6 vs Extended11
# -----------------------
plot_acc=paired.sort_values("Extended11_Accuracy")
fig,ax=plt.subplots(figsize=(8,5))
ypos=np.arange(len(plot_acc))
h=0.35
ax.barh(ypos-h/2,plot_acc["Core6_Accuracy"],height=h,label="Core-6")
ax.barh(ypos+h/2,plot_acc["Extended11_Accuracy"],height=h,label="Extended-11")
ax.set_yticks(ypos,plot_acc["Model"])
ax.set_xlabel("Accuracy")
ax.set_title("Core-6 vs Extended-11 Accuracy")
ax.legend()
fig.tight_layout()
fig.savefig(
    ROOT/"08_Figures"/"Core6_vs_Extended11_Accuracy.png",
    dpi=300,bbox_inches="tight"
)
plt.close(fig)

plot_auc=paired.sort_values("Extended11_ROC_AUC")
fig,ax=plt.subplots(figsize=(8,5))
ypos=np.arange(len(plot_auc))
ax.barh(ypos-h/2,plot_auc["Core6_ROC_AUC"],height=h,label="Core-6")
ax.barh(ypos+h/2,plot_auc["Extended11_ROC_AUC"],height=h,label="Extended-11")
ax.set_yticks(ypos,plot_auc["Model"])
ax.set_xlabel("ROC-AUC")
ax.set_title("Core-6 vs Extended-11 ROC-AUC")
ax.legend()
fig.tight_layout()
fig.savefig(
    ROOT/"08_Figures"/"Core6_vs_Extended11_ROC_AUC.png",
    dpi=300,bbox_inches="tight"
)
plt.close(fig)

# ROC curves for best model across tiers
fig,ax=plt.subplots(figsize=(8,6))
for tier in FEATURE_TIERS:
    prob=oof[tier][BEST_MODEL]
    fpr,tpr,_=roc_curve(y,prob)
    ax.plot(fpr,tpr,label=f"{tier} AUC={roc_auc_score(y,prob):.3f}")
ax.plot([0,1],[0,1],linestyle="--")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"Feature-Tier ROC Curves — {BEST_MODEL}")
ax.legend()
fig.tight_layout()
fig.savefig(
    ROOT/"08_Figures"/"feature_tier_ROC_best_model.png",
    dpi=300,bbox_inches="tight"
)
plt.close(fig)

# -----------------------
# 10) Reproducibility + summary
# -----------------------
config={
    "study":"HeartSense_V3_FeatureSufficiency_Robustness",
    "seed":SEED,
    "outer_folds":OUTER_FOLDS,
    "bootstrap_reps":BOOTSTRAP_REPS,
    "robust_repeats":ROBUST_REPEATS,
    "feature_tiers":FEATURE_TIERS,
    "models":MODEL_NAMES,
    "best_extended_model":BEST_MODEL,
    "external_validation":False,
    "ecg_iot_fusion":False
}
with open(ROOT/"09_Reproducibility"/"experiment_config.json","w") as f:
    json.dump(config,f,indent=2)

(ROOT/"09_Reproducibility"/"IMPORTANT_NOTE.txt").write_text(
    "HeartSense V3 tests feature sufficiency and robustness using the same 918-row public dataset.\n"
    "All model comparisons use identical 5-fold outer splits.\n"
    "Every clean performance estimate is based on outer out-of-fold predictions.\n"
    "Robustness perturbations are applied only to held-out outer-test folds.\n"
    "This is internal validation, not external hospital/cohort validation.\n"
    "No ECG/IoT fusion is claimed because no labeled ECG cohort was supplied.\n",
    encoding="utf-8"
)

# Plain-text summary
summary_lines=[
    "HEARTSENSE V3 SUMMARY",
    "=====================",
    f"Rows: {len(df)}",
    f"Best Extended-11 model by ROC-AUC: {BEST_MODEL}",
    "",
    "Pooled clean OOF results:",
    pooled.sort_values(["FeatureTier","ROC_AUC"],ascending=[True,False]).to_string(index=False),
    "",
    "Core-6 vs Extended-11 paired comparison:",
    paired.to_string(index=False),
]
(ROOT/"SUMMARY.txt").write_text("\n".join(summary_lines),encoding="utf-8")

# -----------------------
# 11) ZIP
# -----------------------
zip_path=Path(
    shutil.make_archive(
        str(WORKDIR/"HeartSense_V3_Results_Bundle"),
        "zip",
        root_dir=ROOT
    )
)

print("\n"+"="*76)
print("HEARTSENSE V3 FINISHED")
print("Best Extended-11 model:",BEST_MODEL)
print("\nExtended-11 results:")
print(
    pooled[pooled["FeatureTier"]=="Extended_11"]
    .sort_values("ROC_AUC",ascending=False)
    .to_string(index=False)
)
print("\nResults bundle:")
print(zip_path)
print("="*76)
